<a href="https://colab.research.google.com/github/saadathar759/Python-EDA-project/blob/main/Zameen_com_data_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

In [ ]:
columns = [
    "City",
    "Location",
    "Price",
    "Type",
    "Purpose",
    "Area",
    "Bedrooms",
    "Bathrooms",
    "Kitchens",
    "Description",
    "Built in year",
    "Parking Spaces",
    "Floors"
]

df = pd.read_excel(
    "/content/sample_data/Scraped_Zameen_Data.xlsx",
    usecols=columns
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/sample_data/Scraped_Zameen_Data.xlsx'

In [ ]:
df.head()

1. Problem Statement

What drives property prices in Pakistan?

The real estate market in Pakistan consists of properties that vary significantly in price due to several factors, including city, location, property type, area, number of bedrooms and bathrooms, and available amenities. For buyers and investors, identifying the factors that have the greatest impact on property prices can be challenging because of the large number of property listings and the diversity of available features.

This project aims to analyze property listings scraped from Zameen.com to determine the key factors that influence property prices in Pakistan. Using data preprocessing, cleaning, feature engineering, and exploratory data analysis (EDA), the study will examine how variables such as city, location, property type, area, bedrooms, bathrooms, and other property characteristics affect prices. The insights generated from this analysis will help investors, buyers, and real estate professionals make more informed decisions.

2. Data Understanding & Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
print("Number of Rows and Columns:")
print(df.shape)

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

In [ ]:
duplicate_rows = df.duplicated().sum()
print("Duplicate Rows:", duplicate_rows)

In [ ]:
df = df.drop_duplicates()

In [ ]:
duplicate_rows = df.duplicated().sum()
print("Duplicate Rows:", duplicate_rows)

3. Missing Values Treatment

In [ ]:
total_missing = df.isnull().sum().sum()

print("Total Missing Values:", total_missing)

In [ ]:
missing_values = df.isnull().sum()

print(missing_values)

In [ ]:
missing_df = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage': (df.isnull().sum()/len(df))*100
})

missing_df = missing_df[missing_df['Missing Values'] > 0]

missing_df.sort_values(by='Missing Values', ascending=False)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(12,6))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title("Missing Values Heatmap")
plt.show()

In [ ]:
df['City'] = df['City'].fillna(df['City'].mode()[0])

In [ ]:
df['City']

In [ ]:
df['City'].isnull().sum()

In [ ]:
categorical_columns = [
    'City',
    'Location',
    'Type',
    'Purpose'
]

for col in categorical_columns:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
import numpy as np
import re

def convert_price(price):

    if pd.isna(price):
        return np.nan

    price = str(price)

    # Remove PKR and new lines
    price = price.replace("PKR", "")
    price = price.replace("\n", " ")
    price = price.strip()

    match = re.search(r'([\d.]+)', price)

    if not match:
        return np.nan

    value = float(match.group(1))

    if "Crore" in price:
        value *= 10000000

    elif "Lakh" in price:
        value *= 100000

    elif "Thousand" in price:
        value *= 1000

    return value

In [ ]:
df["Price"] = df["Price"].apply(convert_price)

In [ ]:
df['Price']

In [ ]:
df["Price"].dtype

In [ ]:
def convert_area(area):

    if pd.isna(area):
        return np.nan

    area = str(area).strip()

    match = re.search(r'([\d.]+)', area)

    if not match:
        return np.nan

    value = float(match.group(1))

    if "Marla" in area:
        value *= 272.25

    elif "Kanal" in area:
        value *= 5445

    elif "Sq. Ft." in area:
        value = value

    elif "Square Feet" in area:
        value = value

    return value

In [ ]:
df["Area"] = df["Area"].apply(convert_area)

In [ ]:
df["Area"].head()

In [ ]:
numeric_columns = [
    "Bedrooms",
    "Bathrooms",
    "Kitchens",
    "Parking Spaces",
    "Floors"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [ ]:
df.info()

In [ ]:
numeric_columns = [
    "Price",
    "Area",
    "Bedrooms",
    "Bathrooms",
    "Kitchens",
    "Parking Spaces",
    "Floors"
]

for col in numeric_columns:
    df[col] = df[col].fillna(df[col].median())

In [ ]:
df['Bathrooms'].value_counts(dropna=False)

In [ ]:
df["Purpose"] = df["Purpose"].apply(
    lambda x: "For Rent" if isinstance(x, str) and x.strip() == "For" else x
)

In [ ]:
df["Purpose"].value_counts()

In [ ]:
df["Description"] = df["Description"].fillna("No Description Available")

In [ ]:
df["Built in year"] = df["Built in year"].fillna(df["Built in year"].mode()[0])

In [ ]:
remaining_missing = df.isnull().sum()

print(remaining_missing)

In [ ]:
!pip install fuzzywuzzy python-Levenshtein

In [ ]:
from fuzzywuzzy import process

In [ ]:
df["City"] = df["City"].astype(str).str.strip()

In [ ]:
df["City"] = df["City"].str.title()

In [ ]:
master_cities = sorted(df["City"].unique())

In [ ]:
def standardize_city(city):

    if pd.isna(city):
        return city

    city = city.strip().title()

    # Find closest matching city
    best_match, score = process.extractOne(city, master_cities)

    # Replace only when confidence is extremely high
    if score >= 95:
        return best_match

    return city

In [ ]:
df["City"] = df["City"].apply(standardize_city)

In [ ]:
print(df["City"].unique())

In [ ]:
df["City"].value_counts()

NameError: name 'df' is not defined

In [ ]:
df.dtypes

In [ ]:
text_columns = ["City", "Location", "Type", "Purpose"]

for col in text_columns:
    df[col] = df[col].str.strip()

In [ ]:
df["Location"] = df["Location"].str.replace(",", "", regex=False)

In [ ]:
df['Price'].value_counts()

In [ ]:
df["Area"] = df["Area"].apply(convert_area)

NameError: name 'df' is not defined

In [ ]:
df['Area']

In [ ]:
categorical_columns = [
    "City",
    "Location",
    "Type",
    "Purpose"
]

for col in categorical_columns:
    df[col] = df[col].str.title()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.boxplot(df["Price"].dropna())
plt.title("Boxplot of Property Prices")
plt.ylabel("Price (PKR)")
plt.show()

NameError: name 'df' is not defined

<Figure size 800x500 with 0 Axes>

In [ ]:
Q1 = df["Price"].quantile(0.25)
Q3 = df["Price"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[
    (df["Price"] < lower_bound) |
    (df["Price"] > upper_bound)
]

print("Number of outliers:", len(outliers))

In [ ]:
outliers.sort_values("Price", ascending=False).head(20)

In [ ]:
df["Price_per_sqft"] = df["Price"] / df["Area"]

In [ ]:
df["Total_Rooms"] = (
    df["Bedrooms"] +
    df["Bathrooms"] +
    df["Kitchens"]
)

NameError: name 'df' is not defined

In [ ]:
current_year = 2026

df["Property_Age"] = current_year - df["Built in year"]

In [ ]:
def area_category(area):

    if area < 1361:
        return "Small"

    elif area < 5445:
        return "Medium"

    else:
        return "Large"

df["Area_Category"] = df["Area"].apply(area_category)

In [ ]:
df.isnull().sum()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

plt.style.use('ggplot')

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(df["Price"], bins=30, kde=True)

plt.title("Distribution of Property Prices")
plt.xlabel("Price (PKR)")
plt.ylabel("Number of Properties")

plt.show()

NameError: name 'sns' is not defined

<Figure size 1000x600 with 0 Axes>

In [ ]:
plt.figure(figsize=(10,6))

sns.histplot(df["Area"], bins=30, kde=True, color="green")

plt.title("Distribution of Property Area")
plt.xlabel("Area (Square Feet)")
plt.ylabel("Number of Properties")

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(x="Bedrooms", data=df)

plt.title("Distribution of Bedrooms")

plt.show()

NameError: name 'sns' is not defined

<Figure size 800x500 with 0 Axes>

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(y="Type", data=df, order=df["Type"].value_counts().index)

plt.title("Property Types")

plt.show()

NameError: name 'sns' is not defined

<Figure size 800x500 with 0 Axes>

In [ ]:
plt.figure(figsize=(12,6))

sns.countplot(
    y="City",
    data=df,
    order=df["City"].value_counts().head(10).index
)

plt.title("Top 10 Cities by Property Listings")

plt.show()

In [ ]:
city_price = (
    df.groupby("City")["Price"]
      .mean()
      .sort_values(ascending=False)
      .head(10)
)

plt.figure(figsize=(12,6))

sns.barplot(
    x=city_price.values,
    y=city_price.index
)

plt.title("Average Property Price by City")

plt.xlabel("Average Price")

plt.ylabel("City")

plt.show()

NameError: name 'df' is not defined

In [ ]:
type_price = (
    df.groupby("Type")["Price"]
      .mean()
      .sort_values(ascending=False)
)

plt.figure(figsize=(10,5))

sns.barplot(
    x=type_price.index,
    y=type_price.values
)

plt.xticks(rotation=45)

plt.title("Average Price by Property Type")

plt.show()

NameError: name 'df' is not defined

In [ ]:
bedroom_price = df.groupby("Bedrooms")["Price"].mean()

plt.figure(figsize=(10,5))

sns.barplot(
    x=bedroom_price.index,
    y=bedroom_price.values
)

plt.title("Average Price by Number of Bedrooms")

plt.show()

NameError: name 'df' is not defined

In [ ]:
plt.figure(figsize=(10,6))

sns.scatterplot(
    data=df,
    x="Area",
    y="Price",
    alpha=0.5
)

plt.title("Area vs Property Price")

plt.show()

NameError: name 'sns' is not defined

<Figure size 1000x600 with 0 Axes>

In [ ]:
numeric_df = df.select_dtypes(include=["number"])

plt.figure(figsize=(10,8))

sns.heatmap(
    numeric_df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Heatmap")

plt.show()

NameError: name 'df' is not defined

In [ ]:
top_cities = df["City"].value_counts().head(10).index

plt.figure(figsize=(14,6))

sns.boxplot(
    data=df[df["City"].isin(top_cities)],
    x="City",
    y="Price"
)

plt.xticks(rotation=45)

plt.title("Property Prices by City")

plt.show()

NameError: name 'df' is not defined

In [ ]:
plt.figure(figsize=(12,6))

sns.boxplot(
    data=df,
    x="Type",
    y="Price"
)

plt.xticks(rotation=45)

plt.title("Property Prices by Property Type")

plt.show()

NameError: name 'sns' is not defined

<Figure size 1200x600 with 0 Axes>

In [ ]:
top_cities = df["City"].value_counts().head(10).index

plt.figure(figsize=(14,6))

sns.boxplot(
    data=df[df["City"].isin(top_cities)],
    x="City",
    y="Price"
)

plt.xticks(rotation=45)

plt.title("Property Prices by City")

plt.show()

NameError: name 'df' is not defined

In [ ]:
plt.figure(figsize=(10,6))

sns.boxplot(
    data=df,
    x="Bedrooms",
    y="Price"
)

plt.title("Property Prices by Bedrooms")

plt.show()

NameError: name 'sns' is not defined

<Figure size 1000x600 with 0 Axes>

8. Conclusion & Next Steps

Summary of Learnings

What drives property prices in Pakistan? The clearest answer the data supports is that price is driven more by location scarcity and room configuration than by plot size. Bedroom/bathroom count correlates with price more strongly (0.35) than area does (0.14), and small properties average higher prices than medium ones, a pattern only explainable by location, since it contradicts a pure size based pricing model.

At the city level, Islamabad's premium (PKR 106.4M average vs. Karachi's PKR 67.2M) is best explained by its restricted, sector based land supply rather than by superior underlying demand. Karachi is the larger and more economically active city, yet averages roughly 37% less per property. Karachi, in turn, delivers the strongest rental income relative to its purchase price (~5.9% gross yield) among major cities, while Rawalpindi and Sialkot show the opposite pattern: comparatively high prices unsupported by proportionate rental income, suggesting their valuations lean more on speculative appreciation than on income fundamentals.

In [ ]:
sale = df[df["Purpose"] == "For Sale"].copy()
rent = df[df["Purpose"] == "For Rent"].copy()

print("Sale listings:", len(sale))
print("Rent listings:", len(rent))
print("\nBlended average price (WRONG — mixes sale & rent):", df["Price"].mean())
print("Sale-only average price (CORRECT):", sale["Price"].mean())
print("Rent-only average price (monthly):", rent["Price"].mean())

NameError: name 'df' is not defined

In [ ]:
numeric_cols = sale.select_dtypes(include="number")
correlations = numeric_cols.corr()["Price"].sort_values(ascending=False)
print(correlations)

NameError: name 'sale' is not defined

In [ ]:
area_price = sale.groupby("Area_Category")["Price"].agg(["mean", "median", "count"])
print(area_price)

NameError: name 'sale' is not defined

In [ ]:
top_cities = sale["City"].value_counts().head(10).index
city_price = sale[sale["City"].isin(top_cities)].groupby("City")["Price"].agg(["mean", "median", "count"])
print(city_price.sort_values("mean", ascending=False))

NameError: name 'sale' is not defined

In [ ]:
sale_city = sale.groupby("City")["Price"].mean()
rent_city = rent.groupby("City")["Price"].mean()  # this is monthly rent

yield_table = pd.DataFrame({
    "avg_sale_price": sale_city,
    "avg_monthly_rent": rent_city
}).dropna()

yield_table["annual_rent"] = yield_table["avg_monthly_rent"] * 12
yield_table["implied_gross_yield_%"] = (yield_table["annual_rent"] / yield_table["avg_sale_price"]) * 100

print(yield_table.sort_values("implied_gross_yield_%", ascending=False))

NameError: name 'sale' is not defined

In [ ]:
Q1 = sale["Price"].quantile(0.25)
Q3 = sale["Price"].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

outliers = sale[sale["Price"] > upper_bound]
print("Number of outliers:", len(outliers), "/", len(sale), f"({len(outliers)/len(sale)*100:.1f}%)")
print("Upper bound (PKR):", upper_bound)
print(outliers["Location"].value_counts().head(10))

NameError: name 'sale' is not defined

Suggestions for Stakeholders (Investors)


Yield oriented investors should prioritize Karachi and Lahore, where rental income currently justifies purchase prices more strongly than in Rawalpindi or Sialkot, where prices appear to be running ahead of rental fundamentals.
Appreciation oriented investors should note that Islamabad's structural land scarcity is a genuinely defensible long-term thesis, but should treat its E-7/F-6 sector properties as a separate luxury tier rather than assuming typical Islamabad property behaves the same way.
Budget constrained investors looking for entry level exposure to the market should consider flats, which offer meaningfully lower capital requirements (median PKR 16.8M vs. PKR 29.4M for houses) without requiring exposure to a different city tier.
All investors should be cautious about using branded society prices (DHA, Bahria Town, Citi Housing) which dominate this dataset's supply as a benchmark for general market value, since these developments carry a premium relative to comparable non branded property in the same city.